# Part 1 — Tokenization: How Text Becomes Numbers

### SentencePiece vs tiktoken, explained by building both

<a href="https://colab.research.google.com/github/jino-shaji/transformer-from-scratch/blob/main/notebooks/01_tokenization_sentencepiece_and_tiktoken.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

A neural network cannot read. It multiplies matrices of floating-point numbers.

So before a transformer can do anything with the sentence *"the river bank was steep"*, something has to convert those characters into integers. That something is the **tokenizer**, and it is the most under-discussed component of the entire stack.

It also has real consequences:

- It fixes your **vocabulary size**, which fixes the width of your model's largest matrices.
- It determines your **sequence length**, and attention cost grows with the square of that.
- It decides how gracefully your model handles a word it has never seen.
- If you work with a non-English language, it silently determines **how much you pay per sentence.**

In this notebook we build up from the naive approaches to the two tokenizers that power most production systems: **tiktoken** (OpenAI) and **SentencePiece** (Google).

In [14]:
!pip install -q tiktoken sentencepiece

In [15]:
import tiktoken, sentencepiece as spm, urllib.request, collections, os, textwrap

# Tiny Shakespeare - our working corpus (~1.1 MB of plain text)
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("input.txt"):
    urllib.request.urlretrieve(URL, "input.txt")

text = open("input.txt").read()
print(f"corpus: {len(text):,} characters")
print(text[:180])

corpus: 1,115,394 characters
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First


## 1. The naive option: character-level

Give every distinct character its own integer. This is what we will use in Part 2, because it is simple enough to fit in your head.

In [16]:
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

encode = lambda s: [stoi[ch] for ch in s]
decode = lambda ids: "".join(itos[i] for i in ids)

sample = "the river bank"
print("vocabulary size :", len(chars))
print("encoded         :", encode(sample))
print("decoded         :", decode(encode(sample)))
print("tokens for sample:", len(encode(sample)))

vocabulary size : 65
encoded         : [58, 46, 43, 1, 56, 47, 60, 43, 56, 1, 40, 39, 52, 49]
decoded         : the river bank
tokens for sample: 14


**The trade-off.** A vocabulary of ~65 is wonderfully small, and nothing is ever out-of-vocabulary — every possible string is representable.

But each token carries almost no meaning. The model has to learn that `t`, `h`, `e` in sequence means "the", and it has to learn that from scratch. Worse, sequences get *long*: a 500-word article becomes ~2,500 tokens. Since self-attention cost scales as O(n²), that length is expensive.

## 2. The other extreme: word-level

Split on whitespace and give every distinct word an integer.

In [17]:
words = text.split()
word_vocab = sorted(set(words))
print(f"distinct words: {len(word_vocab):,}")
print(f"tokens for '{sample}': {len(sample.split())}")

# Now the problem:
unseen = ["blockchain", "Bengaluru", "transformers"]
for w in unseen:
    print(f"  is '{w}' in the vocabulary? {w in set(word_vocab)}")

distinct words: 25,670
tokens for 'the river bank': 3
  is 'blockchain' in the vocabulary? False
  is 'Bengaluru' in the vocabulary? False
  is 'transformers' in the vocabulary? False


**The trade-off.** Sequences are now beautifully short and each token is meaningful. But the vocabulary has exploded to tens of thousands — and that is on 1 MB of Shakespeare. Train on the internet and you get millions of entries, which means an embedding matrix with billions of parameters spent entirely on lookup.

And every word not in the training data becomes `<UNK>`. The model is blind to it. `"transformers"` is unknown even though `"transform"` appears — the relationship is thrown away.

## 3. The compromise everyone actually uses: subword tokenization

Keep frequent words whole. Break rare words into meaningful fragments.

> `"unbelievable"` → `["un", "believ", "able"]`

Now `"unbelievable"` is representable even if never seen, because its pieces were seen. Vocabulary stays in the tens of thousands. Sequences stay reasonably short.

The dominant algorithm for finding those pieces is **Byte-Pair Encoding (BPE)**. It is startlingly simple:

1. Start with a vocabulary of individual characters.
2. Count every adjacent pair in the corpus.
3. Merge the most frequent pair into a single new token.
4. Repeat until you hit your target vocabulary size.

That's the whole algorithm. Let's implement it.

In [18]:
def get_pairs(tokens):
    return collections.Counter(zip(tokens, tokens[1:]))

def merge(tokens, pair, new_token):
    out, i = [], 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
            out.append(new_token); i += 2
        else:
            out.append(tokens[i]); i += 1
    return out

# A toy corpus where the pattern is obvious
corpus = "low lower lowest lower lowest lowest slow slower slowest"
tokens = list(corpus)
vocab  = {}

print(f"start: {len(set(tokens))} unique tokens, {len(tokens)} total\n")

for step in range(8):
    pairs = get_pairs(tokens)
    if not pairs:
        break
    best, count = pairs.most_common(1)[0]
    new_token = best[0] + best[1]
    vocab[new_token] = count
    tokens = merge(tokens, best, new_token)
    print(f"merge {step+1}: {best[0]!r} + {best[1]!r} -> {new_token!r}   (seen {count}x)")

print(f"\nfinal sequence ({len(tokens)} tokens):")
print(tokens)

start: 8 unique tokens, 56 total

merge 1: 'l' + 'o' -> 'lo'   (seen 9x)
merge 2: 'lo' + 'w' -> 'low'   (seen 9x)
merge 3: 'low' + 'e' -> 'lowe'   (seen 7x)
merge 4: ' ' + 'lowe' -> ' lowe'   (seen 5x)
merge 5: 's' + 't' -> 'st'   (seen 4x)
merge 6: ' lowe' + 'st' -> ' lowest'   (seen 3x)
merge 7: ' ' + 's' -> ' s'   (seen 3x)
merge 8: ' lowe' + 'r' -> ' lower'   (seen 2x)

final sequence (14 tokens):
['low', ' lower', ' lowest', ' lower', ' lowest', ' lowest', ' s', 'low', ' s', 'lowe', 'r', ' s', 'lowe', 'st']


Look at what it discovered on its own: `low`, `est`, `er`. Nobody supplied a dictionary or a rule about English suffixes. It found the morphology by counting.

That is BPE. Both libraries below are industrial implementations of this idea — plus a great deal of engineering.

---

## 4. tiktoken — OpenAI's tokenizer

**What it is:** the exact BPE tokenizer used by GPT-3.5, GPT-4, and GPT-4o. Written in Rust, wrapped in Python. It is *inference-only* — you load a pre-trained vocabulary, you do not train your own.

**The key design choice:** tiktoken operates on **UTF-8 bytes**, not characters. The base vocabulary is the 256 possible byte values, so *any* string in *any* language is encodable and nothing is ever out-of-vocabulary. This variant is called byte-level BPE.

Use it when you are working with OpenAI models, counting tokens for cost estimation, or trimming context to fit a window.

In [19]:
enc = tiktoken.get_encoding("cl100k_base")   # GPT-4 / GPT-3.5-turbo
print("vocabulary size:", enc.n_vocab)

s = "The river bank was steep."
ids = enc.encode(s)
print("\ntoken ids :", ids)
print("pieces    :", [enc.decode([i]) for i in ids])
print("roundtrip :", repr(enc.decode(ids)))

vocabulary size: 100277

token ids : [791, 15140, 6201, 574, 32366, 13]
pieces    : ['The', ' river', ' bank', ' was', ' steep', '.']
roundtrip : 'The river bank was steep.'


Notice that the leading space is part of the token — `" river"` rather than `"river"`. That is deliberate: it lets the model distinguish a word starting a sentence from the same word mid-sentence, without wasting a token on whitespace.

Now watch how it handles words it was never explicitly given:

In [20]:
for word in ["transformers", "unbelievable", "Thiruvananthapuram", "antidisestablishmentarianism", "gobbledygook"]:
    ids = enc.encode(word)
    print(f"{word:32s} -> {len(ids):2d} tokens  {[enc.decode([i]) for i in ids]}")

transformers                     ->  2 tokens  ['transform', 'ers']
unbelievable                     ->  3 tokens  ['un', 'belie', 'vable']
Thiruvananthapuram               ->  7 tokens  ['Th', 'ir', 'uv', 'anan', 'th', 'apur', 'am']
antidisestablishmentarianism     ->  6 tokens  ['ant', 'idis', 'establish', 'ment', 'arian', 'ism']
gobbledygook                     ->  4 tokens  ['g', 'obbled', 'yg', 'ook']


Nothing fails. Rare words simply cost more tokens.

### The multilingual tax

This is the part worth knowing if you work outside English. The vocabulary was learned from a corpus that is overwhelmingly English, so English gets efficient whole-word tokens while other scripts fall back to fragments — sometimes to individual bytes.

Run this and read the last column carefully.

In [21]:
pairs = [
    ("English",   "The river bank was steep and cold."),
    ("Hindi",     "\u0928\u0926\u0940 \u0915\u093e \u0915\u093f\u0928\u093e\u0930\u093e \u0922\u0932\u0935\u093e\u0902 \u0914\u0930 \u0920\u0902\u0921\u093e \u0925\u093e\u0964"),
    ("Malayalam", "\u0d2a\u0d41\u0d34\u0d2f\u0d41\u0d1f\u0d46 \u0d15\u0d30 \u0d15\u0d41\u0d24\u0d4d\u0d24\u0d3e\u0d2f\u0d24\u0d41\u0d02 \u0d24\u0d23\u0d41\u0d24\u0d4d\u0d24\u0d24\u0d41\u0d2e\u0d3e\u0d2f\u0d3f\u0d30\u0d41\u0d28\u0d4d\u0d28\u0d41."),
    ("Japanese",  "\u5ddd\u5cb8\u306f\u6025\u3067\u51b7\u305f\u304b\u3063\u305f\u3002"),
]

print(f"{'language':10s} {'chars':>6s} {'tokens':>7s} {'tokens/char':>12s}")
print("-" * 40)
for name, sentence in pairs:
    n = len(enc.encode(sentence))
    print(f"{name:10s} {len(sentence):6d} {n:7d} {n/len(sentence):12.2f}")

language    chars  tokens  tokens/char
----------------------------------------
English        34       8         0.24
Hindi          31      33         1.06
Malayalam      41      72         1.76
Japanese       11      15         1.36


The same sentence, the same meaning, wildly different token counts. Since APIs bill per token and context windows are measured in tokens, non-English users pay more and fit less into the same window for identical content. This is a real, measurable cost — not a theoretical concern.

(GPT-4o's newer `o200k_base` encoding improved this considerably. Compare them yourself:)

In [22]:
enc_o200k = tiktoken.get_encoding("o200k_base")   # GPT-4o

print(f"{'language':10s} {'cl100k':>8s} {'o200k':>8s} {'improvement':>13s}")
print("-" * 42)
for name, sentence in pairs:
    a, b = len(enc.encode(sentence)), len(enc_o200k.encode(sentence))
    print(f"{name:10s} {a:8d} {b:8d} {(a-b)/a:12.0%}")

language     cl100k    o200k   improvement
------------------------------------------
English           8        8           0%
Hindi            33       15          55%
Malayalam        72       16          78%
Japanese         15        9          40%


---

## 5. SentencePiece — Google's tokenizer

**What it is:** a tokenizer *trainer*. You give it raw text, it learns a vocabulary and produces a `.model` file you can ship. It powers T5, ALBERT, XLNet, LLaMA, and most multilingual models.

**The key design choice:** SentencePiece treats the input as a raw Unicode stream and **does not assume spaces separate words**. This matters enormously — Japanese, Chinese, and Thai do not use spaces at all, so a tokenizer that pre-splits on whitespace is structurally unable to handle them.

Its trick is to escape the space character itself as `▁` (U+2581) and treat it as an ordinary symbol. Because spaces are preserved in the token stream, decoding is *lossless and reversible* — you concatenate pieces and swap `▁` back for a space. No language-specific detokenization rules.

Use it when you are training your own model, working in a specific domain (medical, legal, code), or handling languages where OpenAI's vocabulary is inefficient.

In [23]:
spm.SentencePieceTrainer.train(
    input="input.txt",
    model_prefix="shakespeare_bpe",
    vocab_size=2000,
    model_type="bpe",
    character_coverage=1.0,   # 1.0 for alphabetic scripts; ~0.9995 for Chinese/Japanese
)

sp = spm.SentencePieceProcessor(model_file="shakespeare_bpe.model")
print("vocabulary size:", sp.get_piece_size())

vocabulary size: 2000


I0000 00:00:1785165633.323664 9780317 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: input.txt
  input_format: 
  model_prefix: shakespeare_bpe
  model_type: BPE
  vocab_size: 2000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differe

In [24]:
s = "The river bank was steep and cold."

print("pieces   :", sp.encode(s, out_type=str))
print("ids      :", sp.encode(s))
print("roundtrip:", repr(sp.decode(sp.encode(s))))
print()
print("unseen word:", sp.encode("gobbledygook", out_type=str))

pieces   : ['▁The', '▁ri', 'ver', '▁ban', 'k', '▁was', '▁st', 'eep', '▁and', '▁cold', '.']
ids      : [111, 481, 137, 932, 1963, 242, 98, 1760, 45, 1544, 1960]
roundtrip: 'The river bank was steep and cold.'

unseen word: ['▁go', 'b', 'ble', 'd', 'y', 'g', 'ook']


Every piece that begins a word carries the `▁` marker. `['▁The', '▁ri', 'ver', '▁ban', 'k', ...]` — the tokenizer knows where words start, so decoding is unambiguous.

Note also that this vocabulary is only 2,000 entries trained on Shakespeare, so it splits `river` into `▁ri` + `ver`. tiktoken, with 100k entries trained on the internet, keeps it whole. **Vocabulary size buys you shorter sequences.**

### BPE vs Unigram

SentencePiece offers a second algorithm: **Unigram**. Rather than greedily merging upward from characters, it starts with a large candidate vocabulary and iteratively *prunes* the pieces that contribute least to the likelihood of the corpus. It is probabilistic, so it can also sample alternative segmentations of the same string — useful as a regularizer during training.

In [25]:
spm.SentencePieceTrainer.train(
    input="input.txt", model_prefix="shakespeare_uni",
    vocab_size=2000, model_type="unigram",
)
sp_uni = spm.SentencePieceProcessor(model_file="shakespeare_uni.model")

for name, model in [("BPE    ", sp), ("Unigram", sp_uni)]:
    print(f"{name}: {model.encode(s, out_type=str)}")

I0000 00:00:1785165633.439320 9780317 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: input.txt
  input_format: 
  model_prefix: shakespeare_uni
  model_type: UNIGRAM
  vocab_size: 2000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enabl

BPE    : ['▁The', '▁ri', 'ver', '▁ban', 'k', '▁was', '▁st', 'eep', '▁and', '▁cold', '.']
Unigram: ['▁The', '▁r', 'ive', 'r', '▁ba', 'n', 'k', '▁was', '▁', 'st', 'eep', '▁and', '▁cold', '.']


I0000 00:00:1785165634.053278 9780317 unigram_model_trainer.cc:644] EM sub_iter=1 size=4863 obj=9.63425 num_tokens=305103 num_tokens/piece=62.7397
I0000 00:00:1785165634.083477 9780317 unigram_model_trainer.cc:644] EM sub_iter=0 size=3646 obj=9.93177 num_tokens=316251 num_tokens/piece=86.7392
I0000 00:00:1785165634.094418 9780317 unigram_model_trainer.cc:644] EM sub_iter=1 size=3646 obj=9.88294 num_tokens=316252 num_tokens/piece=86.7394
I0000 00:00:1785165634.120582 9780317 unigram_model_trainer.cc:644] EM sub_iter=0 size=2734 obj=10.2255 num_tokens=329651 num_tokens/piece=120.575
I0000 00:00:1785165634.130333 9780317 unigram_model_trainer.cc:644] EM sub_iter=1 size=2734 obj=10.1717 num_tokens=329637 num_tokens/piece=120.569
I0000 00:00:1785165634.153764 9780317 unigram_model_trainer.cc:644] EM sub_iter=0 size=2200 obj=10.4566 num_tokens=341287 num_tokens/piece=155.13
I0000 00:00:1785165634.162300 9780317 unigram_model_trainer.cc:644] EM sub_iter=1 size=2200 obj=10.4132 num_tokens=3412

In [26]:
# Unigram can sample different valid segmentations of the same word.
# This is 'subword regularization' - it makes models more robust.
for _ in range(4):
    print(sp_uni.encode("banishment", out_type=str, enable_sampling=True, alpha=0.1, nbest_size=-1))

['▁', 'b', 'an', 'i', 's', 'h', 'm', 'e', 'n', 't']
['▁b', 'a', 'n', 'i', 's', 'h', 'ment']
['▁b', 'an', 'ish', 'men', 't']
['▁b', 'an', 'is', 'h', 'ment']


---

## 6. Which one should you use?

| | **tiktoken** | **SentencePiece** |
|---|---|---|
| Primary role | Inference-only, pre-trained | Trains a vocabulary from your data |
| Algorithm | Byte-level BPE | BPE or Unigram |
| Operates on | UTF-8 bytes | Raw Unicode stream |
| Assumes spaces split words | Yes (regex pre-split) | No |
| Custom vocabulary | No | Yes |
| Used by | GPT-3.5, GPT-4, GPT-4o | T5, ALBERT, XLNet, LLaMA |
| Reach for it when | Counting tokens, budgeting cost, working against OpenAI APIs | Training your own model, domain-specific or non-English corpora |

A practical rule: **if you are calling a model, use its tokenizer. If you are training a model, train a tokenizer.**

## What we covered

- Character-level: tiny vocabulary, no unknowns, painfully long sequences.
- Word-level: short sequences, exploding vocabulary, brittle on unseen words.
- Subword/BPE: the compromise that won — and it is just "merge the most frequent pair, repeatedly".
- tiktoken: fast, byte-level, fixed vocabulary, what OpenAI models actually use.
- SentencePiece: trainable, language-agnostic, lossless, what you want for your own models.
- Tokenization is not neutral. It has a measurable cost that falls unevenly across languages.

**In Part 2** we take the character-level tokenizer from Section 1, build a bigram model that fails in an instructive way, and then fix it by adding self-attention — arriving at a small working GPT.